<a href="https://colab.research.google.com/github/klorenzoe/hotel-booking-ML-project/blob/feat%2F1%2Fdata-quality/hotel_booking_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1 - [Jorge] Recolección y calidad del dato

**Hotel Booking Demand · Universidad de Deusto**

**Objetivos cuando se está realizando la reserva:**

1. Predecir **si se cancelará** (`is_canceled`)
2. Predecir **tarifa media diaria** (`adr`, *average daily rate*: según el paper, la suma de lo facturado por alojamiento
÷ el número de noches de estancia; tabla 1, p. 43)

**Origen de los datos**

- **Paper:** Antonio, de Almeida y Nunes (2019), [*Hotel booking demand datasets*](https://doi.org/10.1016/j.dib.2018.11.126),
  *Data in Brief* 22, 41–49 · CC BY 4.0.
- **Copia usada:** [TidyTuesday 2020-02-11](https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv) (los dos hoteles en un solo CSV).

## 1.1 Preparar el entorno

In [38]:
import hashlib
import json
import platform
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", None)

ROOT = Path.cwd()
RAW_PATH = ROOT / "data" / "raw" / "hotels.csv"
OUT_DIR = ROOT / "outputs" / "01-data-collection-and-quality"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def check(condition, ok, fail):
    """Print ✅ if the condition holds and ⚠️ otherwise."""
    print(f"✅ {ok}" if bool(condition) else f"⚠️ {fail}")


def pct(mask):
    """True values in %."""
    return round(mask.mean() * 100, 1)


def sha256_file(path: Path) -> str:
    """File fingerprint: if a single byte changes, the hash changes."""
    return hashlib.sha256(path.read_bytes()).hexdigest()



## 1.2 ¿El fichero es el correcto?

In [39]:
SOURCE: dict[str, str] = {
    "name": "hotel-booking-demand",
    "doi": "10.1016/j.dib.2018.11.126",
    "license": "CC BY 4.0",
    "url": "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv",
    "expected_sha256": "7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06",
}

# Expected Values
EXPECTED_ROWS = 119_390  # Paper: Abstract (40,060 + 79,330)
EXPECTED_COLS = 32       # Paper: Abstract (31 variables) + hotel (TidyTuesday)

H1_NAME = "Resort Hotel" # Name by Tidytuesday
H1_BOOKINGS = 40_060     # Paper: Abstract
H1_CANCELED = 11_122     # Paper: Table 3

H2_NAME = "City Hotel"   # Name by Tidytuesday
H2_BOOKINGS = 79_330     # Paper: Abstract
H2_CANCELED = 33_102     # Paper: Table 6

PAPER = pd.DataFrame([
    [H1_NAME, H1_BOOKINGS, H1_CANCELED],
    [H2_NAME, H2_BOOKINGS, H2_CANCELED],
], columns=["hotel", "bookings", "canceled"]).set_index("hotel")


# Download raw data only if missing
if not RAW_PATH.exists():
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(SOURCE["url"], RAW_PATH)


# Read all csv as text
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
raw_sha256: str = sha256_file(RAW_PATH)

# Only the canceled bookings
raw_canceled = raw[raw["is_canceled"] == "1"]


per_hotel = pd.DataFrame({
    "bookings": raw["hotel"].value_counts(),
    "canceled": raw_canceled["hotel"].value_counts(),
}).reindex(PAPER.index) # same index order as PAPER


paper_counts_ok: bool = per_hotel.equals(PAPER)
hash_match_ok = raw_sha256 == SOURCE["expected_sha256"]
rows_cols_match_ok = raw.shape == (EXPECTED_ROWS, EXPECTED_COLS)

check(paper_counts_ok, "Bookings and cancellations counts match the paper.", "Counts differ from the paper.")
check(hash_match_ok, "Hash matches.", "Hash differs: the file changed.")
check(rows_cols_match_ok, f"{EXPECTED_ROWS} rows x {EXPECTED_COLS} columns.", f"Unexpected shape: {raw.shape}.")
per_hotel.join(PAPER.add_suffix(" (paper)"))

✅ Bookings and cancellations counts match the paper.
✅ Hash matches.
✅ 119390 rows x 32 columns.


,bookings,canceled,bookings (paper),canceled (paper)
hotel,,,,
Resort Hotel,40060,11122,40060,11122
City Hotel,79330,33102,79330,33102


## 1.3 ¿Qué representa una fila y cada columna?

- Cada fila es **una reserva**.
- El `role` dice para qué sirve cada columna:
    - `feature`: posible entrada del modelo.
    - `target_cls` / `target_reg`: lo que predecimos.
    - `forbidden`: está en el fichero, pero se rellena después de reservar: **nunca** puede ser entrada.
    - `outcome`: el desenlace de la reserva: **nunca** puede ser entrada.



In [40]:
DATA_DICTIONARY = pd.DataFrame([
    ("hotel",                          "categorical", "feature"),
    ("is_canceled",                    "binary",      "target_cls"),
    ("lead_time",                      "count_days",  "feature"),
    ("arrival_date_year",              "date_part",   "feature"),
    ("arrival_date_month",             "date_part",   "feature"),
    ("arrival_date_week_number",       "date_part",   "feature"),
    ("arrival_date_day_of_month",      "date_part",   "feature"),
    ("stays_in_weekend_nights",        "count",       "feature"),
    ("stays_in_week_nights",           "count",       "feature"),
    ("adults",                         "count",       "feature"),
    ("children",                       "count",       "feature"),
    ("babies",                         "count",       "feature"),
    ("meal",                           "categorical", "feature"),
    ("country",                        "categorical", "forbidden"),
    ("market_segment",                 "categorical", "feature"),
    ("distribution_channel",           "categorical", "feature"),
    ("is_repeated_guest",              "binary",      "feature"),
    ("previous_cancellations",         "count",       "feature"),
    ("previous_bookings_not_canceled", "count",       "feature"),
    ("reserved_room_type",             "categorical", "feature"),
    ("assigned_room_type",             "categorical", "forbidden"),
    ("booking_changes",                "count",       "forbidden"),
    ("deposit_type",                   "categorical", "forbidden"),
    ("agent",                          "id_code",     "feature"),
    ("company",                        "id_code",     "feature"),
    ("days_in_waiting_list",           "count_days",  "feature"),
    ("customer_type",                  "categorical", "feature"),
    ("adr",                            "continuous",  "target_reg"),
    ("required_car_parking_spaces",    "count",       "forbidden"),
    ("total_of_special_requests",      "count",       "feature"),
    ("reservation_status",             "categorical", "outcome"),
    ("reservation_status_date",        "date",        "outcome"),
], columns=["column", "analytic_type", "role"])

check(set(DATA_DICTIONARY["column"]) == set(raw.columns), "The dictionary covers all 32 columns.", "Dictionary and file do not match.")
DATA_DICTIONARY["role"].value_counts()

✅ The dictionary covers all 32 columns.


,count
role,
feature,23
forbidden,5
outcome,2
target_cls,1
target_reg,1


**Se crea una copia tipada de los datos.**

- Los datos `raw` no se tocan.

- Se crea `df` como copia **tipada** de `raw` usando los tipos establecidos por `DATA_DICTIONARY`.

- La conversión no puede perder datos: solo los `"NA"` de `children` pueden quedar vacíos.

In [41]:
# Numeric columns, by their type in the data dictionary
NUMERIC_TYPES = ["binary", "count", "count_days", "continuous"]
NUMERIC_COLS: list[str] = DATA_DICTIONARY.query("analytic_type in @NUMERIC_TYPES")["column"].tolist()

df = raw.copy()

# Text to numbers: anything that is not a number becomes empty
df[NUMERIC_COLS] = df[NUMERIC_COLS].apply(pd.to_numeric, errors="coerce")

# Year + month + day → one arrival date
arrival_text = raw["arrival_date_year"] + "-" + raw["arrival_date_month"] + "-" + raw["arrival_date_day_of_month"]
df["arrival_date"] = pd.to_datetime(arrival_text, format="%Y-%B-%d", errors="coerce")

df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

# Only the "NA" sentinel of children may become empty
children_na = int((raw["children"] == "NA").sum())
empty_per_column = df[NUMERIC_COLS].isna().sum()
empty_columns = empty_per_column[empty_per_column > 0].to_dict()

no_data_lost_ok = empty_columns == {"children": children_na}
arrival_dates_ok = df["arrival_date"].notna().all()

check(no_data_lost_ok, f"Only the {children_na} 'NA' of children became empty.", f"Unexpected empty values: {empty_columns}.")
check(arrival_dates_ok, "Every arrival date is valid.", "Some arrival dates could not be built.")

✅ Only the 4 'NA' of children became empty.
✅ Every arrival date is valid.


## 1.4 ¿Hay duplicados?

**Consideraciones**

- **No existe ID de reserva:** los autores eliminan toda identificación de hotel y cliente por temas de privacidad (*Abstract*, p. 41).

- **Duplicado ambiguo:** una fila repetida puede ser un error o dos reservas parecidas (p. ej., de un grupo).

- **Alcance:** esta sección solo mide; el tratamiento se decide en el 02.

In [42]:
# Every row that has at least one exact copy (the first one included)
dup_all = raw.duplicated(keep=False)

# Only the extra copies (the first one counts as the original)
dup_extra = raw.duplicated()

pd.Series({
    "extra copies (%)": pct(dup_extra),
    "cancel rate, rows with copies (%)": pct(df.loc[dup_all, "is_canceled"] == 1),
    "cancel rate, unique rows (%)": pct(df.loc[~dup_all, "is_canceled"] == 1),
}, name="value").to_frame()


,value
extra copies (%),26.8
"cancel rate, rows with copies (%)",58.4
"cancel rate, unique rows (%)",26.2


**Conclusiones**

- **Muchas copias:** el `26.8 %` de las filas sobra (`dup_extra`); son copias exactas de otra fila.
- **Las copias se cancelan más:** `is_canceled = 1` en el `58.4 %` de las filas con copias (`dup_all`), frente al
  `26.2 %` de las filas únicas (`unique_rows`): más del doble.
- **Riesgo de fuga:** dos copias tienen el mismo `is_canceled`. Si una cae en *train* y la otra en *test*, el
  modelo se evalúa con una fila que ya vio, y las métricas salen mejores de lo real.
- **Implicación:** las filas de un mismo grupo de `dup_all` deben ir al mismo lado de la partición (se decide en
  el 02).

## 1.5 ¿Qué columnas no se pueden usar?

**Consideraciones**

- **Reglas:**
    - El modelo predice al registrar la reserva.
    - Una columna que se rellena después es **fuga**.

- **Momento del dato:**
    - Cada registro es de la **víspera de la llegada**, no del momento de reservar (*Specifications table*, p. 42).
    - Si un valor cambió entre la reserva y la llegada, solo queda el último; es decir, el valor original al reservar no se puede recuperar.

- **Último estado:** Las siguientes variables reflejan el estado final de la reserva (tabla 1, pp. 43-45):
    - `is_canceled`
    - `reservation_status`
    - `reservation_status_date`
    - `days_in_waiting_list`

- **Prohibidas:** Se conocen después de reservar o son el propio desenlace.

- **Dudosas:** Existen al reservar, pero pueden cambiar antes de la llegada (*§2*, pp. 45-46). Se puede probar con y sin ellas, con una lista distinta por tarea.


In [43]:
# Filled in after booking, or the outcome itself → never an input
FORBIDDEN: list[str] = [
    "required_car_parking_spaces",  # data: always 0 in canceled bookings → filled in on arrival
    "booking_changes",              # paper (Table 1, p. 43): counts changes until check-in or cancellation
    "assigned_room_type",           # paper (Table 1, p. 43): can differ from the reserved type; data: set at check-in
    "deposit_type",                 # paper (Table 1, p. 43): depends on payments made until arrival or cancellation
    "country",                      # paper (§2, p. 45): often corrected at check-in; data: PRT far more common among canceled
    "reservation_status",           # paper (Table 1, p. 44): final status = the target
    "reservation_status_date",      # paper (Table 1, p. 45): date the final status was set
]

# Known at booking, but stored as of the day before arrival (paper: Specifications table, p. 42).
UNDER_REVIEW: dict[str, list[str]] = {
    "classification": [
        "stays_in_*_nights",           # paper (§2, p. 46): differ between canceled and kept bookings
        "adults / children / babies",  # paper (§2, p. 46): differ between canceled and kept bookings
        "meal",                        # paper (§2, p. 46): differs between canceled and kept bookings
        "days_in_waiting_list",        # paper (Table 1, p. 43): known only once the booking is confirmed
        "total_of_special_requests",   # requests can be added until arrival
        "adr (as an input)",           # paper (Table 1, p. 43): depends on the final stay
    ],
    "regression": [  # nights, guests and meal set the price → allowed
        "days_in_waiting_list",
        "total_of_special_requests",
    ],
}

# FORBIDDEN must match the roles of the data dictionary
DATA_FORBIDDEN = DATA_DICTIONARY.query("role in ['forbidden', 'outcome']")
COLS_FORBIDDEN = DATA_FORBIDDEN["column"]
forbidden_ok = set(FORBIDDEN) == set(COLS_FORBIDDEN)
check(
    condition=forbidden_ok,
    ok="FORBIDDEN matches the data dictionary roles.",
    fail="FORBIDDEN does not match the data dictionary.",
)

✅ FORBIDDEN matches the data dictionary roles.


In [44]:
canceled = df["is_canceled"] == 1


def by_outcome(mask: pd.Series, name: str) -> pd.DataFrame:
    """Share of True values (%) among arrived and canceled bookings."""
    return pd.DataFrame({"arrived": [pct(mask[~canceled])], "canceled": [pct(mask[canceled])]}, index=[name])

**Parking** (`required_car_parking_spaces`)

In [45]:
# Evidence: parking is filled in on arrival
by_outcome(df["required_car_parking_spaces"] > 0, "with parking (%)")

,arrived,canceled
with parking (%),9.9,0.0


- **Ninguna** reserva cancelada lo tiene (`0.0 %` frente a `9.9 %` de las que llegan).
- El paper no dice cuándo se registra.
- Los datos apuntan a que se anota al llegar; es decir, se prohíbe por los datos, no por el paper.

**Habitación asignada** (`assigned_room_type`)

In [46]:
# Evidence: the room is assigned at check-in
by_outcome(raw["assigned_room_type"] != raw["reserved_room_type"], "assigned room ≠ reserved (%)")

,arrived,canceled
assigned room ≠ reserved (%),18.8,1.8


- El paper dice que puede diferir de la reservada (`reserved_room_type`) por *overbooking* o a petición del cliente.
- Cambia en el `18.8 %` de las reservas que llegan y solo en el `1.8 %` de las canceladas.
- Todo apunta a que se asigna en el *check-in*.

**Cambios en la reserva** (`booking_changes`)

In [47]:
# Evidence: changes are counted until check-in or cancellation
by_outcome(df["booking_changes"] > 0, "with booking changes (%)")

,arrived,canceled
with booking changes (%),20.3,6.4


- El paper los cuenta hasta el *check-in* o la cancelación.
- El `20.3 %` de las que llegan tiene alguno, frente al `6.4 %` de las canceladas.
- Una reserva que acaba de entrar siempre tendría 0.

**Depósito** (`deposit_type`)

In [48]:
# Evidence: the deposit depends on payments made until arrival or cancellation
by_outcome(raw["deposit_type"] == "Non Refund", "Non Refund (%)")

,arrived,canceled
Non Refund (%),0.1,32.8


- El paper lo calcula con los pagos hechos **antes de la llegada o de la cancelación**, no al reservar.
- Por eso `Non Refund` casi solo aparece en canceladas (`32.8 %` frente a `0.1 %`).

**País** (`country`)

In [49]:
# Evidence: the country is often corrected at check-in
by_outcome(raw["country"] == "PRT", "from Portugal (%)")

,arrived,canceled
from Portugal (%),28.0,62.2


- El paper avisa de que la nacionalidad a menudo no se sabe hasta el *check-in*.
- `PRT` aparece en el `62.2 %` de las canceladas y solo en el `28.0 %` de las que llegan.
- Todo apunta a que, mientras no se corrige, el país por defecto es Portugal, y solo las que llegan pasan por el *check-in*.
- Se prohíbe en las dos tareas.

**Desenlace** (`reservation_status`)

In [50]:
# Evidence: reservation_status is the target (Canceled or No-Show → is_canceled = 1)
status_canceled = raw["reservation_status"].isin(["Canceled", "No-Show"])
flag_canceled = raw["is_canceled"] == "1"
target_ok = (status_canceled == flag_canceled).all()
n_no_show = (raw["reservation_status"] == "No-Show").sum()
check(
    condition=target_ok,
    ok=f"is_canceled = 1 ⇔ Canceled or No-Show ({n_no_show:,} no-shows).",
    fail="is_canceled does not match reservation_status.",
)

✅ is_canceled = 1 ⇔ Canceled or No-Show (1,207 no-shows).


- Es el desenlace; es decir, `Canceled` y `No-Show` son `is_canceled = 1`.
- El objetivo es «cancela **o no se presenta**».



---



**Dudosas** (`UNDER_REVIEW`)

- No se prohíben porque el dato existe al reservar.
- Lo que no se sabe es cuánto cambió hasta la víspera de la llegada.
- El comentario de cada una en el código dice por qué es dudosa.

- **La lista depende de la tarea:**
    - **Clasificación:** Noches (`stays_in_weekend_nights`, `stays_in_week_nights`), huéspedes (`adults`, `children`, `babies`) y régimen (`meal`) son dudosas porque se distribuyen distinto entre canceladas y no canceladas; es decir, pueden delatar el desenlace.
    - **Regresión:** Son justo lo que se cotiza (el precio por noche, `adr`, se calcula para esas noches, esas personas y ese régimen). Se admiten como entrada, con la limitación de que el valor guardado es el de la víspera.
    - El 03 medirá cada modelo con y sin sus dudosas.

## 1.6 ¿Cuánta calidad tienen los datos?

**Consideraciones**

- **Dimensiones:** Una pregunta por dimensión:
    - **Completitud** (`completeness`): ¿**Falta** el dato?
    - **Validez** (`validity`): ¿El valor es **posible**?

In [51]:
ADR_MAX: int = 1_000  # domain limit (the 2nd highest adr is 510)
guests = df["adults"] + df["children"].fillna(0) + df["babies"]

quality = pd.DataFrame([
    ("completeness", "adr present (%)",                pct(df["adr"].notna())),
    ("completeness", "country present (%)",            pct(raw["country"] != "NULL")),
    ("validity",     "is_canceled in {0, 1} (%)",      pct(df["is_canceled"].isin([0, 1]))),
    ("validity",     "arrival 2015-07 to 2017-08 (%)", pct(df["arrival_date"].between("2015-07-01", "2017-08-31"))),
    ("validity",     "rows with adr < 0",              int((df["adr"] < 0).sum())),
    ("validity",     f"rows with adr > {ADR_MAX}",     int((df["adr"] > ADR_MAX).sum())),
    ("validity",     "rows without guests",            int((guests == 0).sum())),
    ("validity",     "rows with 0 nights",             int((df["total_nights"] == 0).sum())),
], columns=["dimension", "metric", "value"])
quality

,dimension,metric,value
0,completeness,adr present (%),100.0
1,completeness,country present (%),99.6
2,validity,"is_canceled in {0, 1} (%)",100.0
3,validity,arrival 2015-07 to 2017-08 (%),100.0
4,validity,rows with adr < 0,1.0
5,validity,rows with adr > 1000,1.0
6,validity,rows without guests,180.0
7,validity,rows with 0 nights,715.0


**Lectura**

- **Objetivos:** Están completos y son válidos.

- **Filas imposibles:**
    - Hay 182 (`adr` negativo, `adr` de 5 400, sin huéspedes).
    - Pasan a cuarentena en el 02.

- **Reservas de 0 noches:**
    - Hay 715.
    - `adr` es "por noche"; es decir, sin noches no tiene sentido.
    - Quedan fuera de la regresión.

## 1.7 ¿A quién representan?

In [52]:
pd.Series({
    "City Hotel (%)": pct(raw["hotel"] == "City Hotel"),
    "guests from Portugal (%)": pct(raw["country"] == "PRT"),
    "months covered": df["arrival_date"].dt.to_period("M").nunique(),
}, name="value").to_frame()

,value
City Hotel (%),66.4
guests from Portugal (%),40.7
months covered,26.0


**Lectura**

- **Alcance:** Son dos hoteles portugueses entre 2015 y 2017; es decir, el modelo **no es general**.

- **Estacionalidad:**
    - Son 26 meses: julio y agosto aparecen tres veces.
    - Por eso las medias por mes tiran tienden a temporada de verano.

## 1.8 ¿Pasa el lote al notebook 02?

**Consideraciones**

- **Controles:** Cada control es **PASS** o **FAIL**.

- **Regla:** La decisión depende de qué falla:

| Situación | Decisión |
|---|---|
| Falla un control **crítico** que afecta a todo el `dataset` | `QUARANTINE`: se para |
| Falla otra cosa (filas o columnas concretas) | `CONTINUE_WITH_CONDITIONS` |
| Todo pasa | `PUBLISH` |

In [68]:
q = quality.set_index("metric")["value"]
impossible = (df["adr"] < 0) | (df["adr"] > ADR_MAX) | (guests == 0)

targets_valid_ok = q["is_canceled in {0, 1} (%)"] == 100 and q["adr present (%)"] == 100
period_covered_ok = q["arrival 2015-07 to 2017-08 (%)"] == 100
exact_copies_less_than_5_pct_ok = pct(dup_extra) < 5
zero_nights_ok = q["rows with 0 nights"] == 0

scorecard = pd.DataFrame([
    # control,           scope,      passed,                                              severity,   on failure
    ("paper_counts",      "dataset",  paper_counts_ok,                 "critical", "investigate"),
    ("source_hash",       "dataset",  hash_match_ok,                   "critical", "download again"),
    ("rows_and_columns",  "dataset",  rows_cols_match_ok,              "critical", "investigate"),
    ("targets_valid",     "dataset",  targets_valid_ok,                "critical", "investigate"),
    ("period_covered",    "dataset",  period_covered_ok,               "critical", "investigate"),
    ("exact_copies_<5%",  "dataset",  exact_copies_less_than_5_pct_ok, "warning",  "define a duplicates policy"),  # 5 %: see note below
    ("impossible_rows",   "row",      not impossible.any(),            "critical", "quarantine"),
    ("zero_nights",       "row",      zero_nights_ok,                  "warning",  "exclude from the regression"),
    ("forbidden_columns", "features", len(FORBIDDEN) == 0,             "critical", "exclude from the features"),
], columns=["control", "scope", "passed", "severity", "on_failure"])
scorecard["passed"] = scorecard["passed"].astype(bool)

blocking = scorecard.query("not passed and severity == 'critical' and scope == 'dataset'")
batch_decision: str = "QUARANTINE" if len(blocking) else ("PUBLISH" if scorecard["passed"].all() else "CONTINUE_WITH_CONDITIONS")

print("Batch decision:", batch_decision)

scorecard

Batch decision: CONTINUE_WITH_CONDITIONS


,control,scope,passed,severity,on_failure
0,paper_counts,dataset,True,critical,investigate
1,source_hash,dataset,True,critical,download again
2,rows_and_columns,dataset,True,critical,investigate
3,targets_valid,dataset,True,critical,investigate
4,period_covered,dataset,True,critical,investigate
5,exact_copies_<5%,dataset,False,warning,define a duplicates policy
6,impossible_rows,row,False,critical,quarantine
7,zero_nights,row,False,warning,exclude from the regression
8,forbidden_columns,features,False,critical,exclude from the features


**Lectura**

- **`forbidden_columns`:**
    - Falla **a propósito**, porque las columnas están en el fichero.
    - Se cumple cuando el 03 las deje fuera del modelo.

- **`exact_copies_<5%`:**
    - El 5 % es una convención; es decir, no sale de los datos.
    - Por eso el control es solo un **aviso** (no para el lote).
    - Su efecto es obligar a decidir qué hacer con los duplicados.
    - Con un 26.8 % de copias, el aviso salta con cualquier umbral razonable.

## 1.9 Guardar lo que necesita el notebook 02

- **Contrato** (`data_contract.json`): Qué columnas hay, qué significan y cuáles están prohibidas.

- **Resumen** (`quality_summary.json`): La decisión, las tareas pendientes y con qué se ejecutó.

In [54]:
SENTINELS: dict[str, dict[str, str]] = (
    {  # text codes with a special meaning; "source" = who defines it
        "agent": {
            "value": "NULL",
            "meaning": "not applicable (booking without an agency)",
            "source": "paper (§2, p. 45)",
        },
        "company": {
            "value": "NULL",
            "meaning": "not applicable (booking without a company)",
            "source": "paper (§2, p. 45)",
        },
        "meal": {
            "value": "Undefined",
            "meaning": "same as SC (no meal plan)",
            "source": "paper (Table 1, p. 44)",
        },
        # paper (§2, p. 45): the PMS has no missing data -> reading these as "missing" is a project decision
        "children": {
            "value": "NA",
            "meaning": "missing",
            "source": "our decision"
        },
        "country": {
            "value": "NULL",
            "meaning": "missing",
            "source": "our decision"
        },
        "market_segment": {
            "value": "Undefined",
            "meaning": "missing",
            "source": "our decision",
        },
        "distribution_channel": {
            "value": "Undefined",
            "meaning": "missing",
            "source": "our decision",
        },
    }
)

# Schema: type and role of each column, plus its sentinel and limits
schema: dict[str, dict[str, object]] = {
    r.column: {"type": r.analytic_type, "role": r.role}
    for r in DATA_DICTIONARY.itertuples()
}


for col, sentinel in SENTINELS.items():
    schema[col]["sentinel"] = sentinel


schema["adr"].update(
    {"min": 0, "max": ADR_MAX, "unit": "€/night (currency not stated in the paper)"}
)

identity = {
    "dataset": SOURCE["name"],
    "version": "1.1.0",
    "doi": SOURCE["doi"],
    "license": SOURCE["license"],
    "sha256": raw_sha256,
}
purpose = {
    "targets": {"classification": "is_canceled", "regression": "adr"},
    "prediction_time": "when the booking is registered",
    "data_snapshot": "day before arrival (paper): stored values may differ from those at booking",
    "forbidden_features": FORBIDDEN,
    "features_under_review": UNDER_REVIEW,
}
quality_controls = scorecard[[
    "control",
    "scope",
    "severity",
    "on_failure"
]].to_dict("records")

data_contract: dict[str, object] = {
    "identity": identity,
    "purpose": purpose,
    "schema": schema,
    "quality": quality_controls,
}

In [55]:
PENDING_DECISIONS: list[str] = [  # open questions that the notebook 02 must close
    "exact duplicates: drop, group within the split or weight",
    "adr = 0 with nights > 0 (mostly Complementary): exclude from the regression or not",
    "CN versus CHN: unify or not (compare both populations)",
    "sentinels: NULL in agent/company = 'none'; NULL in country = 'unknown'; Undefined in meal = SC",
]

n_quarantine = int(impossible.sum())
n_zero_nights = int((df["total_nights"] == 0).sum())
environment = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "seed": SEED,
}

quality_summary: dict[str, object] = {
    "dataset": SOURCE["name"],
    "decision": batch_decision,
    "rows_to_quarantine": n_quarantine,
    "rows_excluded_from_regression": n_zero_nights,
    "pending_decisions_for_02": PENDING_DECISIONS,
    "executed_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "environment": environment,
}

In [67]:
def save_json(name: str, obj: dict) -> None:
    """Write obj as readable UTF-8 JSON in OUT_DIR."""
    (OUT_DIR / name).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8"
    )

DATA_CONTRACT_FILENAME="data_contract.json"
QUALITY_SUMMARY_FILENAME="quality_summary.json"
SCORECARD_FILENAME="scorecard.csv"

save_json(DATA_CONTRACT_FILENAME, data_contract)
save_json(QUALITY_SUMMARY_FILENAME, quality_summary)

scorecard.to_csv(OUT_DIR / SCORECARD_FILENAME, index=False)

print(
    "Saved in",
    OUT_DIR.relative_to(ROOT).as_posix(),
    f"-> \n  - {DATA_CONTRACT_FILENAME} \n  - {QUALITY_SUMMARY_FILENAME} \n  - {SCORECARD_FILENAME}",
)

# Summary: the list as bullets, the rest as a table (environment.* flattened)
print("\nPending decisions for 02:")

for item in PENDING_DECISIONS:
    print(f"  - {item}")


print("\n", json.dumps(quality_summary, indent=2, ensure_ascii=False))

Saved in outputs/01-data-collection-and-quality -> 
  - data_contract.json 
  - quality_summary.json 
  - scorecard.csv

Pending decisions for 02:
  - exact duplicates: drop, group within the split or weight
  - adr = 0 with nights > 0 (mostly Complementary): exclude from the regression or not
  - CN versus CHN: unify or not (compare both populations)
  - sentinels: NULL in agent/company = 'none'; NULL in country = 'unknown'; Undefined in meal = SC

 {
  "dataset": "hotel-booking-demand",
  "decision": "CONTINUE_WITH_CONDITIONS",
  "rows_to_quarantine": 182,
  "rows_excluded_from_regression": 715,
  "pending_decisions_for_02": [
    "exact duplicates: drop, group within the split or weight",
    "adr = 0 with nights > 0 (mostly Complementary): exclude from the regression or not",
    "CN versus CHN: unify or not (compare both populations)",
    "sentinels: NULL in agent/company = 'none'; NULL in country = 'unknown'; Undefined in meal = SC"
  ],
  "executed_at": "2026-10-23T15:48:21+00:

**Siguiente:** El paso `2. EDA y Limpieza` lee estos dos JSON, explora los datos y aplica las decisiones pendientes.




---



# 2 - [Mauro] EDA y limpienza

# 3 - [Kathe] Transformación y feature engineering